In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Load the data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. Split features and labels
X = train.drop(columns=['label']).values / 255.0  # Normalize pixel values
y = train['label'].values
X_test = test.values / 255.0

# 3. Create a local validation set to test accuracy before submitting
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

# 4. Train a Random Forest
print("Training Random Forest Classifier...")
clf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
clf.fit(X_train, y_train)

# 5. Check local score
val_preds = clf.predict(X_val)
print(print(f"Local Validation Accuracy: {accuracy_score(y_val, val_preds) * 100:.2f}%"))

# 6. Predict on the official Kaggle test set
print("Generating test predictions...")
test_preds = clf.predict(X_test)

# 7. Create submission file matching sample_submission.csv formatting
submission = pd.DataFrame({
    'ImageId': np.arange(1, len(test_preds) + 1),
    'Label': test_preds
})

submission.to_csv('submission.csv', index=False)
print("Saved baseline 'submission.csv' successfully!")


Training Random Forest Classifier...
Local Validation Accuracy: 94.31%
None
Generating test predictions...
Saved baseline 'submission.csv' successfully!


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Load data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. Extract features and scale pixels to [0, 1]
X_train_raw = train.drop(columns=['label']).values / 255.0
y_train = train['label'].values
X_test_raw = test.values / 255.0

# 3. Reshape flat 784 arrays into 28x28 grayscale image tensors
X_train = X_train_raw.reshape(-1, 28, 28, 1)
X_test = X_test_raw.reshape(-1, 28, 28, 1)

# 4. Build a Convolutional Neural Network (CNN)
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2), # Prevents overfitting
    layers.Dense(10, activation='softmax') # 10 classes for digits 0-9
])

# 5. Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 6. Train the model using a 10% validation split
print("Training CNN...")
model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

# 7. Generate probabilities and extract the highest confidence class
print("Predicting test data...")
probabilities = model.predict(X_test)
test_preds = np.argmax(probabilities, axis=1)

# 8. Match sample_submission.csv format precisely
submission = pd.DataFrame({
    'ImageId': np.arange(1, len(test_preds) + 1),
    'Label': test_preds
})

submission.to_csv('submission.csv', index=False)
print("Saved deep learning 'submission.csv' successfully!")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training CNN...
Epoch 1/5
591/591 ━━━━━━━━━━━━━━━━━━━━ 34s 53ms/step - accuracy: 0.9290 - loss: 0.2317 - val_accuracy: 0.9717 - val_loss: 0.0813
Epoch 2/5
591/591 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.9789 - loss: 0.0697 - val_accuracy: 0.9857 - val_loss: 0.0449
Epoch 3/5
591/591 ━━━━━━━━━━━━━━━━━━━━ 33s 56ms/step - accuracy: 0.9843 - loss: 0.0492 - val_accuracy: 0.9864 - val_loss: 0.0430
Epoch 4/5
591/591 ━━━━━━━━━━━━━━━━━━━━ 41s 55ms/step - accuracy: 0.9874 - loss: 0.0384 - val_accuracy: 0.9886 - val_loss: 0.0394
Epoch 5/5
591/591 ━━━━━━━━━━━━━━━━━━━━ 41s 55ms/step - accuracy: 0.9906 - loss: 0.0283 - val_accuracy: 0.9895 - val_loss: 0.0390
Predicting test data...
875/875 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step
Saved deep learning 'submission.csv' successfully!


In [3]:
# Check your final output structural integrity
sub_check = pd.read_csv('submission.csv')
print(sub_check.head())


   ImageId  Label
0        1      2
1        2      0
2        3      9
3        4      9
4        5      3
